# 01 — Fetch Energy and Capacity Data

**Pipeline stage:** raw inputs.

This is a narrated walkthrough of the first pipeline stage: getting the two raw ingredients every
later stage depends on.

1. **Electricity data** (hourly load, generation, and renewable share of load) for 19 European
   countries, from the public [Energy-Charts API](https://api.energy-charts.info/) — fetched by
   `scripts/fetch_era_energy_inputs.py`.
2. **Renewable-facility capacity maps** (which grid cells actually contain installed wind/solar
   capacity, and how much), from Global Energy Monitor's Global Wind/Solar Power Trackers — built by
   `scripts/import_gem_capacity.py`.

**Teaching note:** neither step needs a paid API key. Cells that only touch `--help` output or the
shared `weather_informed.regions` registry run for real, right now, in this notebook. Cells marked
**DATA CELL — not run here** would fetch real data over the network (slow, rate-limited, and would
leave large files in this repo) — the exact command is shown instead.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from weather_informed.regions import COUNTRIES, EUROPE_CODES, BBOX

print(f"{len(EUROPE_CODES)} countries in the study:\n")
for code_ in EUROPE_CODES:
    print(f"  {code_:3s}  {COUNTRIES[code_]:20s}  bbox={BBOX[code_]}")

19 countries in the study:

  dk   Denmark               bbox=(54.5, 57.8, 7.5, 15.2)
  ie   Ireland               bbox=(51.4, 55.4, -10.6, -5.9)
  nl   Netherlands           bbox=(50.7, 53.6, 3.0, 7.2)
  pt   Portugal              bbox=(36.9, 42.2, -9.6, -6.2)
  gr   Greece                bbox=(34.8, 41.8, 19.3, 28.3)
  be   Belgium               bbox=(49.5, 51.5, 2.5, 6.4)
  lt   Lithuania             bbox=(53.9, 56.5, 20.9, 26.9)
  hr   Croatia               bbox=(42.4, 46.6, 13.5, 19.4)
  bg   Bulgaria              bbox=(41.2, 44.2, 22.4, 28.6)
  lv   Latvia                bbox=(55.7, 58.1, 21.0, 28.2)
  si   Slovenia              bbox=(45.4, 46.9, 13.4, 16.6)
  rs   Serbia                bbox=(42.2, 46.2, 18.8, 23.0)
  sk   Slovakia              bbox=(47.7, 49.6, 16.8, 22.6)
  de   Germany               bbox=(47.2, 55.2, 5.5, 15.5)
  es   Spain                 bbox=(35.7, 43.9, -9.7, 4.5)
  fr   France                bbox=(41.2, 51.3, -5.5, 9.8)
  at   Austria               bbox=(

**Why 19, and why two groups?** The registry splits countries into a `CORE_CODES` set (the original
13) and an `EXPANSION_CODES` set (6 more added later) that together make `EUROPE_CODES`. It also keeps
a `"tx"` (Texas) bounding box that is *not* part of the European study — a non-European sanity-check
region used elsewhere in the project, not one of the 19 countries reported in the paper.

In [2]:
from weather_informed.regions import CORE_CODES, EXPANSION_CODES

print("Core countries:     ", CORE_CODES)
print("Expansion countries:", EXPANSION_CODES)
print()
print("Non-European check region also defined in BBOX:", "tx" in BBOX, "->", BBOX.get("tx"))

Core countries:      ('dk', 'ie', 'nl', 'pt', 'gr', 'be', 'lt', 'hr', 'bg', 'lv', 'si', 'rs', 'sk')
Expansion countries: ('de', 'es', 'fr', 'at', 'cz', 'ro')

Non-European check region also defined in BBOX: True -> (25.8, 36.5, -106.6, -93.5)


## Step 1: electricity data (`fetch_era_energy_inputs.py`)

Fetches hourly `Load` and `Renewable_share_of_load` (plus wind/solar generation columns) from
Energy-Charts, in three eras: pre-COVID (2016–2019), the COVID transition (2020–2021), and post-COVID
(2022–2026). It chunks requests (90 days at a time, ~2s between calls, 8 retries on HTTP 429) so a
full multi-year, 19-country fetch doesn't hammer the API or die on one bad request.

It also does the data-quality correction described in the paper's Methodology: an hourly load value
more than 70% below its centered 336-hour (14-day) rolling median is treated as a sensor/reporting
glitch, replaced by linear interpolation, and the renewable-share numerator is rescaled to keep the
*implied* renewable generation (`share × load / 100`) unchanged — exactly the paper's
`Y_t^corrected = Y_t^original × L_t^original / L_t^interpolated` (Equation for load correction).

In [3]:
!python ../scripts/fetch_era_energy_inputs.py --help

usage: fetch_era_energy_inputs.py [-h]
                                  [--eras {pre,covid,post} [{pre,covid,post} ...]]
                                  [--codes {dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro} [{dk,ie,nl,pt,gr,be,lt,hr,bg,lv,si,rs,sk,de,es,fr,at,cz,ro} ...]]
                                  [--out-dir OUT_DIR] [--audit AUDIT]
                                  [--chunk-days CHUNK_DAYS]
                                  [--retries RETRIES]
                                  [--request-delay REQUEST_DELAY] [--force]
                                  [--dry-run]

Fetch hourly Energy-Charts target inputs for the COVID-era analysis. Only
energy/target columns are written. The spatial-resolution evaluator joins
these files to independently staged ERA5 weather, so downloading a second
flat-city weather product would be redundant.

options:
  -h, --help            show this help message and exit
  --eras {pre,covid,post} [{pre,covid,post} ...]
  --codes {dk,ie,nl,

**DATA CELL — not run here.** A real fetch for every country and era looks like:

```bash
python scripts/fetch_era_energy_inputs.py --eras pre covid post --codes at be bg hr cz dk fr de gr ie lt lv nl pt ro rs sk si es --audit
```

This writes `data/energy_targets_source_by_era/weather_energy_merged_<code>_<era>.csv` per
country/era, plus a coverage audit at `results/covid_era_spatial_resolution/input_coverage.csv`.

## Step 2: capacity maps (`import_gem_capacity.py`)

Converts Global Energy Monitor's wind and solar facility trackers (Excel workbooks, not redistributed
in this repo — see the script's docstring for expected filenames) into one CSV per country per
calendar year: `lat, lon, wind_mw, solar_mw`. A facility counts as active for a year if it was
operating and (per the paper) facilities with no reported start year are included in every year by
default, with a documented sensitivity check available for excluding them instead.

**Why this matters — capacity weighting.** Once we know *where* the wind and solar capacity actually
sits, weather at those specific locations can be weighted more heavily than weather over land with no
installed capacity at all. This is the paper's central spatial-aggregation idea. Here's the worked
example from the paper's Methodology, reproduced exactly:

In [4]:
# Two grid cells for one hour. Cell A is windier AND has almost all the capacity;
# cell B is calmer and has almost none.
wind_speed    = {"A": 10.0, "B": 4.0}    # m/s
wind_capacity = {"A": 900.0, "B": 100.0}  # MW

uniform_avg = sum(wind_speed.values()) / len(wind_speed)
capacity_weighted_avg = (
    sum(wind_speed[c] * wind_capacity[c] for c in wind_speed) / sum(wind_capacity.values())
)

print(f"Uniform average:           {uniform_avg:.1f} m/s")
print(f"Capacity-weighted average: {capacity_weighted_avg:.1f} m/s")

assert abs(uniform_avg - 7.0) < 1e-9
assert abs(capacity_weighted_avg - 9.4) < 1e-9
print("\nMatches the paper's worked example: uniform 7.0 m/s vs. capacity-weighted 9.4 m/s.")
print("90% of the capacity sits in the windier cell, so the capacity-weighted value should — and")
print("does — sit much closer to cell A's 10.0 m/s than the simple average does.")

Uniform average:           7.0 m/s
Capacity-weighted average: 9.4 m/s

Matches the paper's worked example: uniform 7.0 m/s vs. capacity-weighted 9.4 m/s.
90% of the capacity sits in the windier cell, so the capacity-weighted value should — and
does — sit much closer to cell A's 10.0 m/s than the simple average does.


In [5]:
!python ../scripts/import_gem_capacity.py --help

usage: import_gem_capacity.py [-h] [--wind-xlsx WIND_XLSX]
                              [--solar-xlsx SOLAR_XLSX]
                              [--codes {at,be,bg,cz,de,dk,es,fr,gr,hr,ie,lt,lv,nl,pt,ro,rs,si,sk,tx} [{at,be,bg,cz,de,dk,es,fr,gr,hr,ie,lt,lv,nl,pt,ro,rs,si,sk,tx} ...]]
                              [--years YEARS [YEARS ...]]
                              [--unknown-start-policy {include,exclude}]
                              [--solar-dc-to-ac SOLAR_DC_TO_AC]
                              [--no-below-threshold-wind] [--out-dir OUT_DIR]
                              [--results-dir RESULTS_DIR]
                              [--compat-year COMPAT_YEAR]
                              [--compat-dir COMPAT_DIR]

Build annual geolocated wind/solar capacity maps from GEM workbooks.

options:
  -h, --help            show this help message and exit
  --wind-xlsx WIND_XLSX
  --solar-xlsx SOLAR_XLSX
  --codes {at,be,bg,cz,de,dk,es,fr,gr,hr,ie,lt,lv,nl,pt,ro,rs,si,sk,tx} [{at,be,bg,c

**DATA CELL — not run here.** A real import for the post-COVID era looks like:

```bash
python scripts/import_gem_capacity.py --codes at be bg hr cz dk fr de gr ie lt lv nl pt ro rs sk si es \
    --years 2022 2023 2024 2025 2026 --unknown-start-policy include
```

This writes `data/capacity_weights_post_by_year/<year>/<code>.csv` plus audit tables under
`results/post_covid_spatial_resolution/`.

**Next:** [02 — Stage and Aggregate Weather](02_stage_and_aggregate_weather.ipynb) turns raw ERA5
weather into the same per-country, per-year format so it can be combined with these capacity maps.